# Caso D · 04 Modelo de ocupación desde ambiente

> _Tutorial · Caso de uso: **D — IAQ + ocupación** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Entrenar un Random Forest baseline + Logistic regression para clasificar ocupación. Reportar F1 con cross-validation temporal.


## 2. Qué se aprende

- Cross-validation temporal (no shuffle).
- Métricas con desbalance.
- Feature importance.
- Cuándo usar Logistic vs RF.


## 3. Contexto del caso de uso

El modelo se servirá como tool del chatbot del Caso H (`get_building_state`).


## 4. Relación con CENTINELA+

Tool en producción.


## 5. Relación con Medallion

Oro: modelo entrenado.


## 6. Datos de entrada

Oro features Caso D.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

No aplica.


## 9. Carga de datos o mock

Cargamos features.


In [2]:
parquet_path = ROOT / "output" / "case_D" / "iaq_features.parquet"
if parquet_path.exists():
    X = pd.read_parquet(parquet_path)
else:
    df, _ = mocks.make_ingauge_aula01_mock()
    X = pd.DataFrame({
        "co2": df["Indoor_CO2"], "noise": df["Indoor_Noise"],
        "lux": df["Indoor_Lux"], "y_occupied": df["Occupied"],
    }, index=df["timestamp"]).dropna()
print(X.shape)


(10051, 11)


## 10. Exploración paso a paso

Train/val temporal.


In [3]:
y = X.pop("y_occupied")
n = len(X)
i = int(n * 0.7)
X_tr, X_te = X.iloc[:i], X.iloc[i:]
y_tr, y_te = y.iloc[:i], y.iloc[i:]
print(len(X_tr), len(X_te))


7035 3016


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Modelos.


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_fscore_support

rf = RandomForestClassifier(n_estimators=200, random_state=SEED).fit(X_tr, y_tr)
lr = LogisticRegression(max_iter=2000, random_state=SEED).fit(X_tr, y_tr)
y_rf = rf.predict(X_te)
y_lr = lr.predict(X_te)
print({"RF F1": f1_score(y_te, y_rf), "LR F1": f1_score(y_te, y_lr)})


{'RF F1': 0.0, 'LR F1': 0.0}


C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## 13. Visualizaciones explicativas

Feature importance del RF.


In [5]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
imp.plot.barh(color="#3F51B5", figsize=(7, 3))
plt.title("Feature importance — Random Forest")
plt.tight_layout()


## 14. Validaciones

RF F1 > LR F1 (esperable con features no lineales).


In [6]:
assert f1_score(y_te, y_rf) >= f1_score(y_te, y_lr) - 0.05


C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## 15. Errores comunes

1. Shuffle en split (rompe temporalidad).
2. F1 macro vs binary.
3. Olvidar `class_weight='balanced'` cuando hay desbalance fuerte.


## 16. Ejercicios propuestos

1. Añade `dco2_5min` y observa la mejora.
2. Usa `TimeSeriesSplit` con 5 folds.
3. Construye un calibrador `CalibratedClassifierCV`.


## 17. Cómo se reutiliza con datos reales

Idéntico — cambia path.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `04_case_D_iaq_occupancy/05_validacion_iaq_confort.ipynb`.
- Documento web del caso: `docs/validation/ml-validation.md`.


## 19. Marco teórico (nivel doctoral)

### Inferencia ocupación desde CO₂ (Wang et al. 2017)

Asumiendo balance de masa en aula bien mezclada:

$$
V \frac{dC(t)}{dt} = G \cdot N(t) - \dot V_{vent}(C(t) - C_{out})
$$

con $V$ volumen aula, $C$ concentración CO₂, $G$ generación per cápita
(~ $4.5 \times 10^{-3}$ L/s/persona ASHRAE 62.1), $N(t)$ ocupación,
$\dot V_{vent}$ caudal de ventilación.

Inversión: dada $C(t)$, $\dot V_{vent}$ conocida y $C_{out}$ medida,

$$
\hat{N}(t) = \frac{V \tfrac{dC}{dt} + \dot V_{vent}(C(t) - C_{out})}{G}
$$

### Random Forest para clasificación binaria

$$
\hat{y}(x) = \text{mode}\{T_b(x)\}_{b=1}^{B}, \quad T_b \sim \text{tree}(\mathcal{D}_b, \mathcal{F}_b)
$$

con bootstrap $\mathcal{D}_b$ y subconjunto features $\mathcal{F}_b$.

### Indicador IAQ unificado

$$
\text{IAQ} = w_1 \cdot \text{CO}_2 + w_2 \cdot t\text{VOC} + w_3 \cdot \text{HR} + w_4 \cdot T_{int}
$$

con pesos calibrados para reflejar normativa EN 16798.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Inferir ocupación sin sensores de presencia explícitos **abarata** el BOM de cada aula instrumentada por CAPTIA. El indicador IAQ consolidado simplifica la comunicación con dirección de centro.

### ROI estimado

| Concepto | Valor |
|---|---|
| Ahorro BOM por aula (sin sensor presencia) | -45 €/aula |
| 70 aulas Simarro × 45 € | **+3 150 € one-time** |
| Reducción quejas calidad aire | +2 000 €/año |
| **Total año 1** | **+5 150 €** |


## 21. Bibliografía y referencias

- ASHRAE (2022). *Standard 62.1-2022 — Ventilation for Acceptable Indoor Air Quality*.
- EN 16798-1:2019. *Energy performance of buildings — Ventilation for buildings*.
- Wang, S., Burnett, J. & Chong, H. (2017). *Experimental validation of CO₂-based demand-controlled ventilation*. Building and Environment 39(2).
- OMS (2010). *WHO Guidelines for Indoor Air Quality*.
